# L75 gate 전달성 · Kaggle T4×2 · GRT Server 재개

Kaggle **Internet·T4×2**를 켜고 Secrets에 write 권한 `GRT_SERVER_TOKEN`과 `WANDB_API_KEY`를 등록한 뒤 이 노트북의 접근을 허용합니다. 서버는 `https://grt.grtserver.cloud`입니다.

- 최초 실행: `execution_mode='new'`, M·source를 선택하고 **Save Version → Save & Run All**로 실행합니다. 한 Save는 한 job만 실행합니다.
- 12시간 종료 후: 이전 Save의 실행 종료를 확인하고 서버 해당 job에서 **이어서 학습**을 선택합니다. 같은 노트북을 새 Save로 실행하되 `execution_mode='resume'`, `resume_job_id`만 지정합니다. **처음부터 재시도**·새 job 제출은 재개가 아닙니다.
- 첫 셀 실행 시각부터11시간을 worker 종료 시각으로 고정해12시간 강제 종료 전 여유를 둡니다. 셀 재실행으로 연장하지 않습니다. 서버 snapshot은 학습 update 경계에서5분 간격으로 업로드되며, 반드시 서버에서 **verified snapshot**을 확인합니다.

최종 선택은 candidate입니다. 전달성 대조는 **M8 candidate / M8 independent / M75 candidate / M75 independent** 네 조건이며 각40k, A(10,1e-4), write dropout0, seed54, global256, FP32, warmup5%를 유지합니다. 기존20k 안정화 결과는 이40k 기준을 대체하지 않습니다. 실제 결과는 기존 [경로 비교 보고서](https://github.com/YeoJune/GRT/blob/main/docs/experiments/20261019_l75_gate_paths.md)에 추가합니다.

In [ ]:
from pathlib import Path
from datetime import datetime, timedelta, timezone
import hashlib, json, os, re, subprocess, sys

execution_mode = 'new'           # new | resume
num_registers = 8                # 최초 실행만: 8 | 75
write_source = 'candidate'       # 최초 실행만: candidate | independent
resume_job_id = ''               # resume: 서버의 job_... ID
prepared_cache_sha256 = None     # 최초 실행만: 이미 올린 데이터의 manifest SHA256, 없으면 None
campaign = 'l75-gate-transfer-40k-seed54'
server_url = 'https://grt.grtserver.cloud'

if execution_mode not in ('new', 'resume'):
    raise ValueError('execution_mode는 new 또는 resume입니다.')
if execution_mode == 'resume' and not re.fullmatch(r'job_[A-Za-z0-9_-]+', resume_job_id):
    raise ValueError('재개할 서버 job ID를 지정하세요.')
if execution_mode == 'new' and (num_registers not in (8, 75) or write_source not in ('candidate', 'independent')):
    raise ValueError('전달성 비교는 M8/M75 × candidate/independent입니다.')

work_dir = Path('/kaggle/working/grt_gate_transfer')
work_dir.mkdir(parents=True, exist_ok=True)
clock_file = work_dir / 'session_started_utc.json'
if not clock_file.exists():
    clock_file.write_text(json.dumps(datetime.now(timezone.utc).isoformat()))
session_started = datetime.fromisoformat(json.loads(clock_file.read_text()))
server_deadline = (session_started + timedelta(hours=11)).isoformat()
if datetime.now(timezone.utc) >= datetime.fromisoformat(server_deadline):
    raise RuntimeError('이 세션의11시간 worker 종료 시각을 지났습니다. 새 Save에서 재개하세요.')
print('Worker UTC deadline:', server_deadline, '(이 세션에서 고정)')


## 코드 준비·서버 확인

최초 실행은 현재 main을 사용하고 full commit을 서버 job에 고정합니다. 재개는 최신 main으로 학습하지 않고 **기존 job의 `spec.code_ref`를 checkout**합니다. Worker는 소스 hash까지 검사합니다. 재개에서 위 M·source·예산을 수정해도 서버의 고정 조건을 덮어쓰지 않습니다. 실행 중인 다른 작업과 checkout을 공유하지 마세요.

In [ ]:
repo = Path('/kaggle/working/GRT')
if not (repo / '.git').is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/YeoJune/GRT.git', str(repo)], check=True)
if subprocess.check_output(['git', '-C', str(repo), 'status', '--porcelain'], text=True).strip():
    raise RuntimeError('GRT checkout에 수정이 있습니다. 별도 깨끗한 checkout을 사용하세요.')
subprocess.run(['git', '-C', str(repo), 'fetch', 'origin', 'main'], check=True)
sys.path.insert(0, str(repo / 'src'))
from grt.utils.server import ServerClient
client = ServerClient(server_url, secret_source='kaggle')
client.check()

selected_job = None
if execution_mode == 'resume':
    selected_job = client.request('GET', '/jobs/' + resume_job_id)
    if selected_job['status'] != 'queued' or not selected_job.get('resume_artifact_id'):
        raise RuntimeError('이전 Save 종료 후 서버에서 이어서 학습을 선택해 checkpoint 재개 대기 상태로 만드세요.')
    commit = selected_job['spec']['code_ref']
    if not re.fullmatch(r'[a-f0-9]{40}', commit):
        raise ValueError('서버 job의 full commit SHA가 올바르지 않습니다.')
    subprocess.run(['git', '-C', str(repo), 'fetch', 'origin', commit], check=True)
else:
    commit = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'origin/main'], text=True).strip()
subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', commit], check=True)
print('GRT full commit:', commit)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(repo) + '[wandb,wikitext]'], check=True)

# 학습·사후 분석은 설치된 지원 버전을 사용하는 별도 Python 프로세스에서 실행합니다.
from grt.utils.kaggle import WikiTextRun
from grt.utils.server import code_identity
if selected_job is not None:
    if (*code_identity(repo), 2) != (selected_job['spec']['code_ref'], selected_job['spec']['code_sha256'], selected_job['spec']['world_size']):
        raise RuntimeError('원래 code/source hash/world_size=2와 다릅니다. worker를 실행하지 않습니다.')
print('서버 write 권한·코드 준비 완료. 토큰 값은 출력하지 않습니다.')


## 최초 실행만: 데이터 캐시·job 등록

서버에 prepared 데이터를 반드시 보관한 뒤 job을 등록합니다. 같은 조건의 제출을 셀 재실행으로 중복 생성하지 않도록 조건과 commit의 hash를 멱등 키로 사용합니다. 출력되는 **job ID·commit·manifest hash**를 보관하세요. `resume`에서는 데이터 준비·새 제출을 모두 건너뜁니다.

In [ ]:
if execution_mode == 'new':
    queue_name = f'{campaign}-m{num_registers}-{write_source}'
    # 제출용 객체는 설정·데이터만 준비하고, 실제 학습은 아래 서버 queue worker가 실행합니다.
    submission = WikiTextRun(
        repo_dir=repo, work_dir=work_dir, runtime='kaggle', world_size=2,
        size='small', budget='40k', models=('grt',), tokenization='gpt2_bpe',
        num_registers=num_registers, router_hidden=512, seed=54, wandb=True,
        test_at_end=False, server_mode='off', server_url=server_url,
        secret_source='kaggle', server_queue=queue_name,
        overrides={
            'model': {'segment_len': 75,
                      'router': {'write_source': write_source, 'write_detach_source': False},
                      'register': {'conditional_read': False, 'write_gate_bias_init': 0.0,
                                   'dropout_prob': 0.0, 'write_logit_threshold': 10.0,
                                   'write_logit_coef': 1e-4}},
            'training': {'auto_batch': True, 'memory_fraction': 0.95, 'warmup_ratio': 0.05},
            'rtla': {'enabled': True}},
    )
    submission.setup(install=False)
    submission.prepare_data(server_cache=prepared_cache_sha256)
    cached = submission.cache_prepared()
    print('Prepared manifest SHA256:', cached['extensions']['manifest_sha256'])
    client.ensure_schema()
    client.ensure_queue(queue_name)
    body = submission.server_job('grt', posthoc_options={
        'batch_size': 8, 'segments': 8, 'warmup_segments': 3, 'horizon': 4,
        'slots': True, 'registers': True, 'window_starts': [0, 16, 32], 'group_repeats': 3})
    body['name'] = f'L75 M{num_registers} {write_source} 40k seed54'
    body['labels']['campaign'] = campaign
    request_id = 'gate-transfer-' + hashlib.sha256(json.dumps(body, sort_keys=True, separators=(',', ':')).encode()).hexdigest()
    client.request('POST', '/jobs/validate', body)
    submitted = client.request('POST', '/jobs', body, key=request_id)
    selected_job = client.request('GET', '/jobs/' + submitted['id'])
else:
    queue_name = selected_job['queue']

print('Job ID:', selected_job['id'])
print('Job URL:', server_url)
print('Status:', selected_job['status'], 'queue:', selected_job['queue'])
print('M / source / target steps:', selected_job['spec']['config']['model']['num_registers'],
      selected_job['spec']['config']['model']['router']['write_source'], selected_job['spec']['target_steps'])
(work_dir / 'selected_job.json').write_text(json.dumps({
    'job_id': selected_job['id'], 'code_ref': selected_job['spec']['code_ref'],
    'dataset_sha256': selected_job['spec']['dataset_sha256'], 'queue': selected_job['queue']}, indent=2))


## 학습 worker: 최초·재개 공통

`--once`로 이 조건의 전용 큐에서 하나만 실행합니다. 새 Save에서는 새 로컬 디렉터리에 서버의 last/best·optimizer·scheduler·rank별 RNG·stream 위치·메모리 state·W&B ID를 복원합니다. **재개 시 preflight 자동 배치를 다시 측정하지 않고 기존 micro/accum을 유지**하므로 같은 T4×2를 사용하세요.

일반 `job.preflight()`·`job.train()` 셀을 별도로 실행하지 않고 worker에 맡깁니다. 40k와 지정한 posthoc가 모두 서버 검증에 통과하면 `succeeded`입니다. 중단되면 같은 job에서 다시 **이어서 학습**을 선택합니다.

In [ ]:
selected_job = client.request('GET', '/jobs/' + selected_job['id'])
if selected_job['status'] != 'queued':
    raise RuntimeError('새 worker는 queued job에서만 시작합니다. 완료/실행 중 job을 중복 실행하지 마세요.')
if execution_mode == 'resume' and not selected_job.get('resume_artifact_id'):
    raise RuntimeError('Checkpoint 재개가 아닙니다. 서버에서 이어서 학습을 선택하세요.')
# 큐가 전용인지 확인해 다른 조건을 대신 가져오는 일을 막습니다.
from urllib.parse import quote
queued = client.request('GET', '/jobs?queue=' + quote(selected_job['queue'], safe='') + '&status=queued&limit=2')
if [item['id'] for item in queued['data']] != [selected_job['id']] or queued.get('next_cursor'):
    raise RuntimeError('이 전용 큐에 다른 대기 job이 있습니다. 서버에서 큐를 분리하세요.')
if datetime.now(timezone.utc) >= datetime.fromisoformat(server_deadline):
    raise RuntimeError('세션 worker 종료 시각을 지났습니다. 새 Save에서 실행하세요.')

from kaggle_secrets import UserSecretsClient
read_secret = UserSecretsClient().get_secret
worker_env = dict(os.environ, GRT_SERVER_URL=server_url,
                  GRT_SERVER_TOKEN=read_secret('GRT_SERVER_TOKEN'),
                  WANDB_API_KEY=read_secret('WANDB_API_KEY'))
try:
    result = subprocess.run([
        sys.executable, str(repo / 'scripts/server_worker.py'),
        '--work-dir', str(work_dir / 'worker'), '--prepared-from-server',
        '--deadline', server_deadline, '--runtime', 'kaggle', '--secret-source', 'env',
        '--queue', selected_job['queue'], '--device', 'cuda', '--world-size', '2', '--once',
    ], env=worker_env, check=False)
finally:
    del worker_env
print('Worker exit code:', result.returncode)
print('서버 job 상태·verified resume snapshot을 확인하세요:', selected_job['id'])


## 서버 저장 확인·새 Save 재개

아래 셀은 비밀값 없이 상태와 verified snapshot의 step만 표시합니다. 학습 중에는 서버 웹에서도 확인할 수 있습니다. 연결/파일 검증이 실패하면 마지막 **검증 완료한 snapshot**까지만 보존됩니다. 5분 이후 진행분이나 강제 삭제 순간의 저장은 보장되지 않습니다.

1. 이전 Kaggle Save가 실제 종료됐는지 확인합니다. Stale running이면 lease 만료까지 기다린 뒤 웹에서 **이어서 학습**을 누르고 종료 근거를 입력합니다. `interrupted/failed/cancelled`는 종료 근거 없이 재개할 수 있습니다.
2. 다음 Save에서 `execution_mode='resume'`, 같은 `resume_job_id`를 입력하고 T4×2·Internet·Secrets를 유지한 채 **Save & Run All** 합니다. 새 job을 만들거나 예산을 남은 step 수로 줄이지 않습니다.
3. 복원 로그의 step이0이 아닌지, 같은 job의 attempt 수가 증가했는지 확인합니다. 서버가 Kaggle Save 자체를 자동으로 만들지는 않습니다.
4. 최종 `succeeded` 후 서버에서 학습·posthoc 원본을 수집합니다. 결과 ZIP과 학습 조건을 전달하면 기존 실험 보고서에 기록합니다.

**최초 실행 네 조건:** M8/candidate → M8/independent → M75/candidate → M75/independent. 각각 첫 Save는 `new`, 그 실험의 추가 Save는 `resume`입니다. 이전40k matched baseline이 확보돼 있다면 해당 independent 실행은 생략할 수 있습니다.

In [ ]:
detail = client.request('GET', '/jobs/' + selected_job['id'])
snapshots = [a for a in detail['artifacts'] if a['status'] == 'verified'
             and a.get('extensions', {}).get('resume_format') == 'grt.resume/1']
print('Job:', detail['id'], 'status:', detail['status'], 'attempts:', len(detail['attempts']))
print('Verified resume steps:', sorted(a['extensions']['step'] for a in snapshots))
if detail['status'] != 'succeeded' and not snapshots:
    print('아직 검증 완료 snapshot이 없습니다. 서버 저장 오류·로그를 확인하세요.')
